# 01 — Exploratory analysis

Goal: understand the study shape and the label source *before* any modelling.

Data: the frozen simulated snapshot in `data/raw/simulated_seed2026/`.
**Simulated data only — never report these numbers as findings.**

This notebook imports `focustrack`; it does not define features or labels.

In [1]:
from pathlib import Path

import pandas as pd

from focustrack.config import EVENT_ONLY_COLUMNS, load_config
from focustrack.io import load_source

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

cfg = load_config(ROOT / "configs" / "c1_proposed.yaml")
sessions, epochs, probes = load_source("simulated", cfg)
cfg.name

'c1_proposed'

In [2]:
pd.DataFrame(
    {
        "participants": [sessions["participant_id"].nunique()],
        "sessions": [sessions["session_id"].nunique()],
        "epochs": [len(epochs)],
        "probes": [len(probes)],
        "screen_sessions": [int((sessions["study_mode"] == "screen").sum())],
        "non_screen_sessions": [int((sessions["study_mode"] == "non-screen").sum())],
    }
)

,participants,sessions,epochs,probes,screen_sessions,non_screen_sessions
0,22,132,6600,574,64,68


## Probes and labelling

Each probe rates the previous five minutes. The observer column exists in
library/campus sessions and is used for the agreement audit, never as a feature.

In [3]:
probe_view = probes.merge(
    sessions[["session_id", "study_mode", "task"]], on="session_id", how="left"
)
display(probe_view.groupby("study_mode")["rating"].agg(["count", "mean", "std"]).round(2))
print(f"observer audited: {probe_view['observer'].notna().mean():.1%} of probes")

,count,mean,std
study_mode,,,
non-screen,296,2.40,0.80
screen,278,2.65,0.85


observer audited: 64.3% of probes


## Feature availability

The simulated snapshot (and the PP1 logger) stores per-epoch aggregates only.
Event-only features are `NaN` here; that gap is documented in `docs/DECISIONS.md`.

In [4]:
features_path = ROOT / "data" / "processed" / "features.csv"
if not features_path.exists():
    raise FileNotFoundError("run `make data` first")

features = pd.read_csv(features_path)
display(features[list(EVENT_ONLY_COLUMNS)].isna().mean().sort_values(ascending=False).to_frame("missing_rate"))

,missing_rate
inter_key_mean_ms,1.0
burst_count,1.0
mean_burst_duration_s,1.0
mean_burst_length,1.0
click_rate,1.0
unique_apps,1.0
app_category_entropy,1.0


## Before modelling

- Check that every participant has probes and that no session has zero labelled epochs.
- Check class balance of `focus_label` overall and per study mode.
- Check that non-screen sessions are not treated as "idle = distracted"; the
  component only claims interruption detection there (§3.8).
- Confirm all normalisation is fitted on training participants only.